### Models for wine dataset

In [72]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import TargetEncoder


In [73]:
wines = pd.read_csv("wines_inferred.csv")

# quick fill of a column
wines.loc[wines["region_name"].isna(), "region_name"] = "Kakheti"

#### Testing tree based models

Models: 
- Random Forest
- XGBoost
- CatBoost

Evaluation metrics for classification and regression
- MAE/MSE
- F1 score, accuracy

The models will predict the ratings (and eventually the price). 

WARNING: food and grapes_merged have to be extended

-> currency translation ?

---

### Dataset spliting 

In [74]:
wines.columns

Index(['id', 'id_parent', 'seo_name', 'name', 'type', 'year', 'country',
       'region_name', 'winery_name', 'base_volume', 'price', 'base_currency',
       'intensity', 'sweetness', 'acidity', 'tannin', 'fizziness',
       'mean_rating', 'n_rating', 'ratings_distribution', 'pro_rating', 'food',
       'alcohol', 'grapes_merged', 'drinking_window'],
      dtype='object')

In [99]:
X_cols = ['seo_name', 'name', 'type', 'year', 'country',
       'region_name', 'winery_name', 'base_volume', 'price', 'base_currency',
       'intensity', 'sweetness', 'acidity', 'tannin', 'fizziness', 
       'alcohol', 'drinking_window'] # 'grapes_merged' 'food'

y_cols = ['mean_rating'] # 'n_rating', 'ratings_distribution', 'pro_rating'

cat_cols = ['type','country',
       'region_name', 'winery_name', 'base_currency']

# try adding n_rating

---

### Split of the dataset


####

In [100]:
# splits, stratified by country

X_train, X_test, y_train, y_test = train_test_split(
                                        wines[X_cols], wines[y_cols], test_size=0.2, random_state=42,
                                        stratify=wines["country"]
                                    )
work_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis = 1)
# Final X_test and y_test

---

## Regression models

### Random Forests

As RF needs encoding, we will need to encode the categorical columns:
- names : 49897 val
- type OHE : 7 values
- country can OHE : 21 values
- region : 1429 value
- winery_name : 8543 val
- base_currency can OHE : 8 values 

In [101]:
# functions for some preproc

def ohe_col(df, col: str, drop = True):
    """apply OHE to column col, with drop or not"""
    ohe = OneHotEncoder(sparse_output = False) 
    encoded = ohe.fit_transform(df[[col]])  # note double brackets to keep it 2D
    df_encoded = pd.DataFrame(encoded, columns=ohe.get_feature_names_out([col]), dtype=int)
    if drop:
        return pd.concat([df.drop(columns=[col]), df_encoded], axis=1)
    return pd.concat([df, df_encoded], axis=1)

def target_col(df_train, y_train, col): # folds for less data leak
    """fit and apply target encoding for col on train"""
    df = df_train.copy()
    
    enc_auto = TargetEncoder(smooth="auto", random_state=42)
    col_trans = enc_auto.fit_transform(df[[col]], y_train.values.flatten())
    enc_auto.handle_unknown = 'value'  # options: 'return_nan', 'value', 'ignore'
    enc_auto.handle_missing = 'value'
    enc_auto.fill_value = y_train.mean()
    df[[col]] = col_trans
    return df, enc_auto

def apply_target_col(df_test, col, encoder):
    """apply the target encoding on test"""
    df = df_test.copy()
    col_trans = encoder.transform(df[[col]])
    df[[col]] = col_trans
    return df

def train_test_metric_reg(y_train, preds_train, y_test, preds_test):
    """print regression mae mse metrics for train and test"""
    r2_train = r2_score(y_train, preds_train)
    r2_test = r2_score(y_test, preds_test)
    
    mae_train = mean_absolute_error(y_train.values, preds_train)
    mae_test = mean_absolute_error(y_test.values, preds_test)

    mse_train = mean_squared_error(y_train.values, preds_train)
    mse_test = mean_squared_error(y_test.values, preds_test)

    print(f"Metrics on train : MAE = {mae_train} MSE = {mse_train} R2 = {r2_train}")
    print(f"Metrics on test : MAE = {mae_test} MSE = {mse_test} R2 = {r2_test}")
    

In [102]:
# preprocessing
df = work_df[X_cols + y_cols].copy().drop(columns=["seo_name", "name"]) # atm drop 

# OHE column 'type'
df = ohe_col(df, "type", drop = True)

# OHE column 'country'
df = ohe_col(df, "country", drop = False)

# OHE column base_currency
df = ohe_col(df, "base_currency", drop = True) # translate to 1 currency?

# target column region
df, region_enc = target_col(df, df[y_cols], "region_name")

# target column winery
df, winery_enc = target_col(df, df[y_cols], "winery_name")

In [103]:
# splits, stratified by country
X, y = df.drop(columns=y_cols), df[y_cols]

X_train_sub, X_test_sub, y_train_sub, y_test_sub = train_test_split(
                                        X, y, test_size=0.2, random_state=42,
                                        stratify=X["country"]
                                    )
# dropping country
X_train_sub = X_train_sub.drop(columns="country")
X_test_sub = X_test_sub.drop(columns="country")

X_train_sub.shape, X_test_sub.shape

((32089, 48), (8023, 48))

In [104]:
from sklearn.ensemble import RandomForestRegressor

regr = RandomForestRegressor(max_depth=2, random_state=0)
regr.fit(X_train_sub, y_train_sub.values.flatten())

RandomForestRegressor(max_depth=2, random_state=0)

In [105]:
preds_train = regr.predict(X_train_sub)
preds = regr.predict(X_test_sub)

_ = train_test_metric_reg(y_train_sub, preds_train, y_test_sub, preds)

Metrics on train : MAE = 0.4761593830734885 MSE = 0.815281885956687 R2 = 0.1787560894913327
Metrics on test : MAE = 0.47556070640629916 MSE = 0.8334437988085628 R2 = 0.18304585325312184


---

### XGBoost Regressor

In [128]:
# preprocessing
df = work_df[X_cols + y_cols].copy().drop(columns=["seo_name", "name"]) # atm drop 

# OHE column 'type'
df = ohe_col(df, "type", drop = True)

# OHE column 'country'
df = ohe_col(df, "country", drop = False)

# OHE column base_currency
df = ohe_col(df, "base_currency", drop = True) # translate to 1 currency?

# target column region
df, region_enc = target_col(df, df[y_cols], "region_name")

# target column winery
df, winery_enc = target_col(df, df[y_cols], "winery_name")

In [129]:
# splits, stratified by country
X, y = df.drop(columns=y_cols), df[y_cols]

X_train_sub, X_test_sub, y_train_sub, y_test_sub = train_test_split(
                                        X, y, test_size=0.2, random_state=42,
                                        stratify=X["country"]
                                    )
# dropping country
X_train_sub = X_train_sub.drop(columns="country")
X_test_sub = X_test_sub.drop(columns="country")

X_train_sub.shape, X_test_sub.shape

((32089, 48), (8023, 48))

In [130]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train_sub, y_train_sub) # eval_seton subset?

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [131]:
preds_train = xgb.predict(X_train_sub)
preds = xgb.predict(X_test_sub)

_ = train_test_metric_reg(y_train_sub, preds_train, y_test_sub, preds)

Metrics on train : MAE = 0.3289117383961595 MSE = 0.4026456736920933 R2 = 0.594409843628172
Metrics on test : MAE = 0.4231196211647065 MSE = 0.6921845264820579 R2 = 0.3215103165541343


---

### CatBoost Regressor

In [132]:
# preprocessing
df = work_df[X_cols + y_cols].copy().drop(columns=["seo_name", "name"]) # atm drop 

df['base_currency'] = df['base_currency'].astype(str) # nan considered like float

In [133]:
# splits, stratified by country
X, y = df.drop(columns=y_cols), df[y_cols]

X_train_sub, X_test_sub, y_train_sub, y_test_sub = train_test_split(
                                        X, y, test_size=0.2, random_state=42,
                                        stratify=X["country"]
                                    )

X_train_sub.shape, X_test_sub.shape

((32089, 15), (8023, 15))

In [134]:
df.columns

Index(['type', 'year', 'country', 'region_name', 'winery_name', 'base_volume',
       'price', 'base_currency', 'intensity', 'sweetness', 'acidity', 'tannin',
       'fizziness', 'alcohol', 'drinking_window', 'mean_rating'],
      dtype='object')

In [135]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    cat_features=cat_cols,
    verbose=100,
    random_state=42,
    allow_writing_files=False
)

model.fit(X_train_sub, y_train_sub)

0:	learn: 0.9872242	total: 13.8ms	remaining: 6.91s
100:	learn: 0.8384636	total: 818ms	remaining: 3.23s
200:	learn: 0.8245292	total: 1.68s	remaining: 2.49s
300:	learn: 0.8137017	total: 2.54s	remaining: 1.68s
400:	learn: 0.8061345	total: 3.34s	remaining: 825ms
499:	learn: 0.7991232	total: 4.16s	remaining: 0us


In [136]:
preds_train = model.predict(X_train_sub)
preds = model.predict(X_test_sub)

_ = train_test_metric_reg(y_train_sub, preds_train, y_test_sub, preds)

Metrics on train : MAE = 0.40305748385444046 MSE = 0.6353753650932289 R2 = 0.35997823764018144
Metrics on test : MAE = 0.42750260740710494 MSE = 0.7053772010784899 R2 = 0.3085786584943533


In [ ]:
# # Visualisation 

# fig, axes = plt.subplots(1, 1, figsize=(6, 5))

# sns.set_theme(style="darkgrid")

# df_plot = pd.DataFrame({
#     "preds": preds,
#     "trues": y_test.values.flatten()
# })

# sns.scatterplot(df_plot, x = "preds", y="trues", ax = axes)
# min_val = min(df_plot["preds"].min(), df_plot["trues"].min())
# max_val = max(df_plot["preds"].max(), df_plot["trues"].max())
# plt.plot([min_val, max_val], [min_val, max_val], color='red', label='y=x')


# plt.legend()
# plt.tight_layout()

---

## Classification models